# Gabarito — Exercícios Módulo 4
### Análise Exploratória de Dados (EDA)
**Curso Introdutório de Python para Ciência de Dados | T326 - UNIFOR**

---

In [ ]:
# ── Setup ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, io

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120, 'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA', 'axes.grid': True, 'grid.alpha': 0.4,
})
sns.set_palette('Set2')

def carregar_brazilian_cities(caminho):
    with open(caminho, 'r', encoding='utf-8', newline='') as f:
        raw = f.read()
    linhas = raw.split('\r\n')
    def fix(l):
        if l.startswith('"') and l.endswith('"'):
            l = l[1:-1]
        return l.replace('""', '"')
    conteudo = '\n'.join([fix(l) for l in linhas if l.strip()])
    return pd.read_csv(io.StringIO(conteudo))

df = carregar_brazilian_cities('../../modulo-2/datasets/brazilian_city.csv')
df = df.rename(columns={
    'IDHM Ranking 2010': 'IDHM_Ranking', 'IBGE_CROP_PRODUCTION_$': 'IBGE_CROP_PROD',
    'WAL-MART': 'WALMART', 'IBGE_1-4': 'IBGE_1a4', 'IBGE_5-9': 'IBGE_5a9',
    'IBGE_10-14': 'IBGE_10a14', 'IBGE_15-59': 'IBGE_15a59', 'IBGE_60+': 'IBGE_60mais',
})
df['POP_FINAL']  = df['IBGE_RES_POP'].where(df['IBGE_RES_POP'] > 0, df['ESTIMATED_POP'])
df['DENSIDADE']  = np.where(df['AREA'] > 0, (df['POP_FINAL'] / df['AREA']).round(2), np.nan)
df['TIPO']       = df['CAPITAL'].map({1: 'Capital', 0: 'Interior'})
df['IDHM_FAIXA'] = pd.cut(df['IDHM'],
    bins=[0, 0.499, 0.599, 0.699, 0.799, 1.0],
    labels=['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto'])
regioes = {
    'Norte': ['AM','PA','RR','RO','AC','AP','TO'],
    'Nordeste': ['MA','PI','CE','RN','PB','PE','AL','SE','BA'],
    'Centro-Oeste': ['MT','MS','GO','DF'],
    'Sudeste': ['SP','RJ','MG','ES'],
    'Sul': ['PR','SC','RS']
}
df['REGIAO'] = df['STATE'].map({uf: reg for reg, ufs in regioes.items() for uf in ufs})
ordem_reg = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
print(f"✅ Dataset pronto: {df.shape[0]:,} municípios × {df.shape[1]} variáveis")

---
## Exercício 1 — Exploração inicial

In [ ]:
# ── Exercício 1: Exploração inicial ─────────────────────────────
total = len(df)

# 1a) Municípios por região com percentual
print("📌 Municípios por Região:")
por_regiao = df['REGIAO'].value_counts().reindex(ordem_reg)
for reg, n in por_regiao.items():
    barra = '█' * int(n / 100)
    print(f"  {reg:<15} {n:>5,}  ({n/total*100:.1f}%)  {barra}")

# 1b) Top 3 variáveis com mais nulos
print("\n📌 Top 3 variáveis com mais valores ausentes:")
nulos = df.isnull().sum().sort_values(ascending=False)
for col, n in nulos.head(3).items():
    print(f"  {col:<30} {n:>5} nulos ({n/total*100:.1f}%)")

# 1c) Coeficiente de variação do IDHM por estado
print("\n📌 Heterogeneidade interna do IDHM por Estado (CV%):")
cv_estado = df.groupby('STATE')['IDHM'].agg(['std','mean']).dropna()
cv_estado['CV'] = (cv_estado['std'] / cv_estado['mean'] * 100).round(2)
cv_estado = cv_estado.sort_values('CV', ascending=False)
print(f"  Mais heterogêneo: {cv_estado.index[0]} (CV = {cv_estado.iloc[0]['CV']:.1f}%)")
print(f"  Mais homogêneo:   {cv_estado.index[-1]} (CV = {cv_estado.iloc[-1]['CV']:.1f}%)")
print("\n  Top 5 estados mais heterogêneos:")
for uf, row in cv_estado.head(5).iterrows():
    print(f"    {uf}: CV = {row['CV']:.1f}%")

---
## Exercício 2 — Análise univariada

In [ ]:
# ── Exercício 2: Painel com 4 distribuições ──────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Distribuições — Variáveis Socioeconômicas', fontsize=14, fontweight='bold')

variaveis = [
    ('IDHM',         lambda x: x,          'IDHM',           '#4C72B0'),
    ('IDHM_Educacao',lambda x: x,          'IDHM Educação',  '#C44E52'),
    ('GDP_CAPITA',   np.log10,             'log₁₀(PIB per Capita)', '#55A868'),
    ('DENSIDADE',    np.log10,             'log₁₀(Densidade)', '#8172B2'),
]

for ax, (col, transform, titulo, cor) in zip(axes.flatten(), variaveis):
    dados = df[col].dropna()
    dados = dados[dados > 0] if col in ['GDP_CAPITA', 'DENSIDADE'] else dados
    dados_t = transform(dados)
    ax.hist(dados_t, bins=35, color=cor, edgecolor='white', alpha=0.85)
    media  = dados_t.mean()
    mediana = dados_t.median()
    ax.axvline(media,   color='black',  linestyle='--', linewidth=1.5, label=f'Média={media:.3f}')
    ax.axvline(mediana, color='orange', linestyle='-.', linewidth=1.5, label=f'Mediana={mediana:.3f}')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_ylabel('Municípios')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Exercício 3 — Análise bivariada

In [ ]:
# ── Exercício 3: Correlação com IDHM ─────────────────────────────
vars_analise = ['GDP_CAPITA', 'IDHM_Educacao', 'IDHM_Renda',
                'IDHM_Longevidade', 'PAY_TV', 'Cars', 'DENSIDADE']

correlacoes = df[['IDHM'] + vars_analise].corr()['IDHM'].drop('IDHM').sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
cores = ['#D32F2F' if v < 0 else '#1565C0' for v in correlacoes.values]
bars = ax.barh(correlacoes.index, correlacoes.values, color=cores, edgecolor='white', height=0.6)

for bar, val in zip(bars, correlacoes.values):
    xpos = val + 0.01 if val >= 0 else val - 0.01
    ha = 'left' if val >= 0 else 'right'
    ax.text(xpos, bar.get_y() + bar.get_height()/2, f'{val:+.3f}',
            va='center', ha=ha, fontsize=10, fontweight='bold')

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlação de Pearson com o IDHM', fontsize=14, fontweight='bold')
ax.set_xlabel('Correlação')
ax.set_xlim(-0.3, 1.1)
plt.tight_layout()
plt.show()

print("\n💡 Variável com maior correlação positiva:", correlacoes.idxmax())
print("💡 Variável com maior correlação negativa:", correlacoes.idxmin())

---
## Exercício 4 — Análise categórica

In [ ]:
# ── Exercício 4: Pivot table + heatmap ───────────────────────────
pivot = pd.crosstab(df['REGIAO'], df['IDHM_FAIXA'])
pivot = pivot.reindex(ordem_reg)

print("📌 Tabela pivot — Quantidade de municípios por Região × Faixa IDHM:")
display(pivot)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Municípios'}, ax=ax)
ax.set_title('Quantidade de Municípios por Região × Faixa de IDHM',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Faixa de IDHM')
ax.set_ylabel('Região')
plt.tight_layout()
plt.show()

print("\n💡 Nordeste tem a maior concentração de municípios na faixa Baixo.")
print("💡 Sul e Sudeste concentram municípios nas faixas Alto e Muito Alto.")

---
## Exercício 5 — Mini-storytelling

In [ ]:
# ── Exercício 5: Dashboard sobre desigualdade entre estados ──────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Desigualdade entre Estados Brasileiros — IDHM',
             fontsize=14, fontweight='bold')

palette_reg = dict(zip(ordem_reg,
    ['#E07B54', '#E0C354', '#54A0E0', '#54E089', '#9B54E0']))

# Painel 1: Boxplot de IDHM por região
sns.boxplot(data=df.dropna(subset=['IDHM','REGIAO']),
            x='REGIAO', y='IDHM', order=ordem_reg,
            palette=[palette_reg[r] for r in ordem_reg],
            width=0.55, fliersize=2, ax=axes[0])
axes[0].set_title('Distribuição por Região', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('IDHM')
axes[0].tick_params(axis='x', rotation=20)

# Painel 2: IDHM médio por UF (top 10 e bottom 10)
idhm_uf = df.groupby(['STATE','REGIAO'])['IDHM'].mean().reset_index()
idhm_uf = idhm_uf.sort_values('IDHM')
top = idhm_uf.tail(7)
bot = idhm_uf.head(7)
combined = pd.concat([bot, top])
cores_uf = [palette_reg.get(r, 'gray') for r in combined['REGIAO']]
axes[1].barh(combined['STATE'], combined['IDHM'], color=cores_uf, edgecolor='white')
axes[1].axvline(df['IDHM'].mean(), color='black', linestyle='--', alpha=0.7,
                label=f"Média BR={df['IDHM'].mean():.3f}")
axes[1].set_title('Top 7 e Bottom 7 Estados', fontweight='bold')
axes[1].set_xlabel('IDHM Médio')
axes[1].legend(fontsize=8)

# Painel 3: Scatter GDP × IDHM por estado
idhm_gdp = df.groupby(['STATE','REGIAO']).agg(
    IDHM=('IDHM','mean'), GDP=('GDP_CAPITA','mean')).reset_index()
for reg in ordem_reg:
    g = idhm_gdp[idhm_gdp['REGIAO'] == reg]
    axes[2].scatter(g['GDP'], g['IDHM'], color=palette_reg[reg], s=60,
                    alpha=0.85, label=reg, edgecolors='white')
    for _, row in g.iterrows():
        axes[2].annotate(row['STATE'], (row['GDP'], row['IDHM']),
                         fontsize=6, ha='center', va='bottom')
axes[2].set_title('PIB per Capita × IDHM\npor Estado', fontweight='bold')
axes[2].set_xlabel('PIB per Capita (R$)')
axes[2].set_ylabel('IDHM Médio')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}k'))
axes[2].legend(fontsize=7, markerscale=1.2)

plt.tight_layout()
plt.show()

print("\n📖 INSIGHTS:")
print("  1. A desigualdade regional no IDHM é estrutural: Norte e Nordeste")
print("     concentram os estados com piores índices, refletindo desigualdades")
print("     históricas em investimentos em educação e infraestrutura.")
print("  2. Estados do Sul (SC, RS, PR) têm IDHM alto com PIB per capita moderado,")
print("     sugerindo que eficiência na conversão de renda em bem-estar importa.")
print("  3. O DF destoa: PIB per capita elevado mas IDHM moderado — o alto PIB")
print("     é concentrado, não se convertendo igualmente em bem-estar geral.")